# 高考文章评分 Agent

In [1]:
from langchain_agent.agents.base import get_singleton_client

advanced_model = get_singleton_client(llm_provider="bailing")

✓ 已初始化百灵客户端，使用模型: Ling-1T


In [2]:
from langgraph.graph import StateGraph, START, END
import re
from langchain.messages import SystemMessage
from pydantic import BaseModel, Field

In [3]:
class ScoreDetail(BaseModel):
    """单维度评分详情"""

    score: float = Field(default=0.0, ge=0, le=1, description="分数 0~1")
    reason: str = Field(default="", description="评分理由")


class EssayState(BaseModel):
    """高考作文评分状态"""

    topic: str = Field(..., description="题目")
    essay: str = Field(..., description="待评作文")
    relevance: ScoreDetail = Field(
        default_factory=ScoreDetail, description="审题立意：是否切题、立意是否深刻"
    )
    evidence: ScoreDetail = Field(
        default_factory=ScoreDetail, description="论据分析：材料是否充实、论据是否有力"
    )
    structure: ScoreDetail = Field(
        default_factory=ScoreDetail, description="结构评估：行文逻辑、段落衔接"
    )
    expression: ScoreDetail = Field(
        default_factory=ScoreDetail, description="语言文采：用词、修辞、句式"
    )
    final_score: float = Field(default=0.0, description="最终加权分数")

In [4]:
llm = advanced_model.with_structured_output(ScoreDetail)

In [5]:
SYSTEM_PROMPT = """你是一名经验丰富的高考作文阅卷老师。请遵循以下准则
          ：
        2 1. 严格按照高考作文评分标准进行评判
        3 2. 客观公正，不受作文主题立场影响
        4 3. 评分理由要具体、有依据，引用作文原文
        5 4. 每个维度独立评分，不因某维度表现好而影响其他维度
        6 5. 评分范围 0~1，0.6 为及格线，0.8 以上为优秀"""

In [6]:
from pydantic import BaseModel


def build_json_prompt(schema: type[BaseModel]) -> str:
    """构建 JSON 输出的系统提示词。

    Args:
        schema: Pydantic 模型类，定义输出的 JSON 结构

    Returns:
        系统提示词字符串

    注意:
    Schema 必须包含 is_valid: bool 字段用于标识是否提取到有效信息，
    其他字段建议为 Optional 类型。

    示例:
        from pydantic import BaseModel, Field
        from typing import Optional, Literal

        class ProductReview(BaseModel):
            ""Analysis of a product review.""
            rating: Optional[int]  = Field(description="The rating of the product", ge=1, le=5)
            sentiment: Optional[Literal["positive", "negative"]] = Field(description="The sentiment of the review")
            key_points: Optional[list[str]] = Field(description="The key points of the review. Lowercase, 1-3 words each.")
            is_valid: bool = Field(description="是否提取到了有效的信息")

        prompt = build_json_prompt(ProductReview)
    """
    schema_name = schema.__name__
    schema_json = schema.model_json_schema()

    return f"""你是一个 JSON 输出助手。
                【输出规则】
                1. 只输出纯 JSON 对象，不要包含任何解释、markdown 代码块或其他内容
                2. 根据输入内容判断：从用户的输入中提取有效的 {schema_name} 信息
                3. 确保 JSON 字段与 Schema 定义完全一致

                【Schema 定义】
                ## {schema_name}
                {schema_json}
                """

In [7]:
def extract_score_and_reason(content: str) -> tuple[float, str]:
    """从 LLM 回复中提取分数和理由。"""
    match = re.search(r"Score:\s*(\d+(\.\d+)?)", content)
    if match:
        score = float(match.group(1))
        reason_start = match.end()
        reason = content[reason_start:].strip().lstrip("。.").strip()
        return score, reason if reason else content[reason_start:].strip()
    raise ValueError(f"无法从回复中提取分数：{content}")


def check_relevance(state: EssayState) -> dict:
    """审题立意：检查是否切题、立意是否深刻。"""
    messages = [
        SystemMessage(build_json_prompt(ScoreDetail) + SYSTEM_PROMPT),
        SystemMessage(
            "请评估以下高考作文的审题立意。"
            "考察是否切题、立意是否深刻。"
            "给出 0 到 1 之间的分数，并说明理由。"
            f"\n\n题目：{state.topic} \n\n作文：{state.essay}"
        ),
    ]
    result = llm.invoke(messages)
    try:
        return {"relevance": result}
    except ValueError as e:
        print(f"check_relevance 错误：{e}")
        return {"relevance": ScoreDetail(score=0.0, reason="评分失败")}


def check_evidence(state: EssayState) -> dict:
    """论据分析：检查材料是否充实、论据是否有力。"""
    messages = [
        SystemMessage(build_json_prompt(ScoreDetail) + SYSTEM_PROMPT),
        SystemMessage(
            "请评估以下高考作文的论据分析。"
            "考察材料是否充实、论据是否有力。"
            "给出 0 到 1 之间的分数，并说明理由。"
            f"\n\n作文：{state.essay}"
        ),
    ]
    result = llm.invoke(messages)
    try:
        return {"evidence": result}
    except ValueError as e:
        print(f"check_evidence 错误：{e}")
        return {"evidence": ScoreDetail(score=0.0, reason="评分失败")}


def check_structure(state: EssayState) -> dict:
    """结构评估：检查行文逻辑、段落衔接。"""
    messages = [
        SystemMessage(build_json_prompt(ScoreDetail) + SYSTEM_PROMPT),
        SystemMessage(
            "请评估以下高考作文的结构。"
            "考察行文逻辑、段落衔接是否合理。"
            "给出 0 到 1 之间的分数，并说明理由。"
            f"\n\n作文：{state.essay}"
        ),
    ]
    result = llm.invoke(messages)
    try:
        return {"structure": result}
    except ValueError as e:
        print(f"check_structure 错误：{e}")
        return {"structure": ScoreDetail(score=0.0, reason="评分失败")}


def check_expression(state: EssayState) -> dict:
    """语言文采：检查用词、修辞、句式。"""
    messages = [
        SystemMessage(build_json_prompt(ScoreDetail) + SYSTEM_PROMPT),
        SystemMessage(
            "请评估以下高考作文的语言文采。"
            "考察用词是否贴切、修辞是否恰当、句式是否灵活。"
            "给出 0 到 1 之间的分数，并说明理由。"
            f"\n\n作文：{state.essay}"
        ),
    ]
    result = llm.invoke(messages)
    try:
        return {"expression": result}
    except ValueError as e:
        print(f"check_expression 错误：{e}")
        return {"expression": ScoreDetail(score=0.0, reason="评分失败")}


def calculate_final_score(state: EssayState) -> dict:
    """根据各维度分数计算加权最终分数。"""
    print(f"state {state}")
    final = (
        state.relevance.score * 0.3
        + state.evidence.score * 0.2
        + state.structure.score * 0.2
        + state.expression.score * 0.3
    )
    return {"final_score": round(final, 2)}


def guodu(state: EssayState) -> dict:
    """根据各维度分数计算加权最终分数。"""
    pass

In [8]:
# Initialize the StateGraph
workflow = StateGraph(EssayState)

# Add nodes to the graph
workflow.add_node("check_relevance", check_relevance)
workflow.add_node("guodu", guodu)
workflow.add_node("check_evidence", check_evidence)
workflow.add_node("check_structure", check_structure)
workflow.add_node("check_expression", check_expression)
workflow.add_node("calculate_final_score", calculate_final_score)


workflow.add_edge(START, "check_relevance")
workflow.add_edge("guodu", "check_evidence")
workflow.add_edge("guodu", "check_structure")
workflow.add_edge("guodu", "check_expression")

workflow.add_edge("check_evidence", "calculate_final_score")
workflow.add_edge("check_structure", "calculate_final_score")
workflow.add_edge("check_expression", "calculate_final_score")
# workflow.add_edge("check_relevance", "calculate_final_score")
workflow.add_edge("calculate_final_score", END)

# Define and add conditional edges
workflow.add_conditional_edges(
    "check_relevance",
    lambda x: "guodu" if x.relevance.score > 0.5 else "calculate_final_score",
)

# Compile the graph
app = workflow.compile()

In [9]:
# return result

In [10]:
topic = """以车轮为喻：“车轮的辐条一根一根，向心辏集，连接起居于中心的轮毂。辐集而轮运，劲直的辐条汇聚于轮毂，车轮支撑起载重的车辆，滚滚向前。”要求考生根据这段话，结合自身体验，展开联想与思考进行写作"""

In [11]:
sample_essay = """
    车轮行进的道理很简单：辐条坚韧挺直，众多辐条向中心汇聚，轮毂才能牢固；轮毂牢固，车轮才能承重，车子才能奔行千里。然而，这道理难道只适用于车轮吗？当然不是，一个国家的发展之路也是如此。你看，当个体拥有百折不回的风骨，就像“辐条”般劲直；当社会拥有万流归海的向心力，就像“轮毂”般聚合，只有这两者相辅相成，才能铸就团结共进的民族脊梁，支撑着国家在文明富强的康庄大道上行稳致远。

国之基石，在于民心有铁骨，这便是“辐聚成轮”的根基所在。能担当大任者，必先端正自身，不为富贵所惑，不因贫贱而移，不向强权而屈；秉持为公之心，不困于蝇头微利，不囿于蜗角虚名，将铮铮铁骨化作滋养民族精神的力量。苏武持节北海，其脊梁历经十九载风霜而不折；鲁迅以笔为刃，在沉沉黑夜中开辟新路，唤醒沉睡的民众；陈祥榕用热血捍卫界碑，将那句“清澈的爱”熔铸成丰碑。正是这一根根“劲直”的脊梁，构成了车轮最坚实的底座。

然而，孤木难成林，单辐不成轮，万千铁骨何以“辐聚成轮”？这是因为中华民族自古就有融汇百川、凝聚万方的“毂”。这“毂”非限于一时一地之政权，而是跨越时空的文化认同与价值归属。从文明初始的多元并存，到秦的大一统，再到唐朝“爱之如一”的胸怀，中华民族的向心力从未中断。正因为此，我们才能看到塞罕坝三代人接续奋斗，将荒漠变为林海；数百家动画公司携手，助力“哪吒”走向世界。千年文明汇聚的向心力，正托举中华民族不断奋进。

辐劲直，毂聚合，则车轮自成。但车轮的承载力更取决于国家与民众的“双向奔赴”。辐条与轮毂实为共生的整体，就像国以民为本，民以国为依。当风雨呼啸时，总有无数肩膀扛起重任，把微光聚成火炬；当人民在异国遭遇危难，国家也会成为最坚实的后盾，万里归途，不弃一人。正是在这种相互依存中，个人价值得以升华，国家之车轮得以滚滚向前。

辐条坚韧如铁骨，轮毂牢固聚丹心，车轮滚滚间铸就了中华民族数千年不倒的脊梁。在这同心同德的巨轮上，我们必将向着民族复兴的辉煌未来一路高歌。
    """

In [12]:
initial_state = EssayState(
    topic=topic,
    essay=sample_essay,
)
# result = app.invoke(initial_state)
# for event in app.stream(initial_state, stream_mode='updates'):
#     print(event)

In [13]:
from langgraph_essay_grading.graph import graph

for event in graph.stream(initial_state, stream_mode="updates"):
    print(event)

{'check_relevance': {'relevance': ScoreDetail(score=0.9, reason="作文精准把握车轮喻体核心内涵——'辐条劲直'象征个体风骨，'轮毂聚合'代表集体向心力，'车轮滚动'指向国家发展进程，立意深刻且完全切题。文中'国之基石，在于民心有铁骨'直接呼应材料'劲直的辐条'，'中华民族自古就有融汇百川、凝聚万方的“毂”'对应'辐集而轮运'，并结合苏武、鲁迅、陈祥榕等实例论证个体精神，又以塞罕坝、动画产业协作阐释集体向心力，最后升华至'国家与民众的双向奔赴'，逻辑层层递进，展现对材料哲理的深度开掘。全文无偏离题意内容，思想高度符合高考优秀作文标准。")}}
{'fan_out': None}
{'check_evidence': {'evidence': ScoreDetail(score=0.85, reason='论据分析较为充实有力。材料选取典型：苏武持节、鲁迅启蒙、陈祥榕戍边等事例具体且具有历史厚重感，有效支撑‘个体铁骨’的分论点；塞罕坝治沙、国产动画崛起等当代案例体现‘向心力’的现实延续，论证层次清晰。引用原文如‘百折不回的风骨’‘铮铮铁骨化作滋养民族精神的力量’‘融汇百川、凝聚万方’等贴合论点展开分析，逻辑链条完整。不足在于对‘双向奔赴’机制的剖析稍显简略，未深入举例说明国家与个体如何具体互动强化承载力，故未达满分。')}}
{'check_structure': {'structure': ScoreDetail(score=0.9, reason='文章结构严谨，逻辑清晰。开篇由车轮喻体自然引出国家发展的核心观点；主体部分层层递进：先论证个体‘铁骨’如辐条（第二段引用苏武、鲁迅、陈祥榕等事例支撑），再阐释民族‘向心力’如轮毂（第三段以历史脉络和塞罕坝、国产动画案例说明文化认同的凝聚作用），最后升华至国家与民众的‘双向奔赴’实现共生共运（第四段通过风雨同担、万里归侨等场景强化互动关系）；结尾回扣车轮意象，呼应主题。全文段落衔接紧密，因果关系明确，喻体贯穿始终且与说理高度契合，符合高考议论文‘起承转合’的高阶结构要求。')}}
{'check_expression': {'expression': ScoreDetail(score=0.85, reason="语言文采表现优秀。用词精准贴切，如'百折不回的